# 02 · Measuring Retrieval

Three numbers, measured separately, because they fail independently.

You will measure index recall against exact search, compare four retrieval
strategies against gold labels, and see the specific questions that single-shot
retrieval cannot answer regardless of strategy.

In [1]:
import json, re, textwrap
import numpy as np, pandas as pd
from pathlib import Path

pd.set_option("display.width", 130); pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd()
while not (ROOT / "data" / "corpus").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CORPUS = ROOT / "data" / "corpus"
DOCS = json.loads((CORPUS / "manifest.json").read_text())
EVAL = json.loads((ROOT / "data" / "corpus_eval.json").read_text())
print(f"{len(DOCS)} documents, {len(EVAL)} eval questions")

10 documents, 10 eval questions


In [2]:
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

chunks = pd.read_csv(ROOT / "data" / "chunks.csv")
vecs = np.load(ROOT / "data" / "chunk_vecs.npy")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
qvecs = encoder.encode([e["question"] for e in EVAL], normalize_embeddings=True)
print(f"{len(chunks)} chunks | {vecs.shape[1]}-dim | {len(EVAL)} questions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

36 chunks | 384-dim | 10 questions


## Strategy 1 — dense vector search

In [3]:
def dense(qv, k=5):
    scores = vecs @ qv
    idx = np.argsort(-scores)[:k]
    return chunks.iloc[idx]["chunk_id"].tolist()

def docs_of(chunk_ids):
    return {c.split("#")[0] for c in chunk_ids}

def recall_at_k(strategy, k=5):
    hits = []
    for e, qv in zip(EVAL, qvecs):
        if not e["gold_docs"]:
            continue                                  # unanswerable, scored separately
        got = docs_of(strategy(qv, k) if strategy is dense else strategy(e, k))
        hits.append(len(set(e["gold_docs"]) & got) / len(e["gold_docs"]))
    return float(np.mean(hits))

print(f"dense recall@5 : {recall_at_k(dense):.3f}")

dense recall@5 : 0.944


## Strategy 2 — BM25 keyword search

Best Matching 25. No model, no index, no GPU — and on identifier-heavy text it
is frequently the stronger of the two.

In [4]:
from rank_bm25 import BM25Okapi

def tok(s): return re.findall(r"[a-z0-9\-]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in chunks["text"]])

def keyword(e, k=5):
    scores = bm25.get_scores(tok(e["question"]))
    return chunks.iloc[np.argsort(-scores)[:k]]["chunk_id"].tolist()

print(f"BM25 recall@5  : {recall_at_k(keyword):.3f}")

BM25 recall@5  : 0.944


## Strategy 3 — hybrid, fused by rank

Reciprocal rank fusion. It uses only positions, so the incomparable scores of the
two systems never need normalising. Two lines.

In [5]:
def rrf(e, k=5, pool=20, kconst=60):
    qv = encoder.encode(e["question"], normalize_embeddings=True)
    d_ids = chunks.iloc[np.argsort(-(vecs @ qv))[:pool]]["chunk_id"].tolist()
    b_ids = chunks.iloc[np.argsort(-bm25.get_scores(tok(e["question"])))[:pool]]["chunk_id"].tolist()
    fused = {}
    for lst in (d_ids, b_ids):
        for rank, cid in enumerate(lst, 1):
            fused[cid] = fused.get(cid, 0) + 1 / (kconst + rank)
    return [cid for cid, _ in sorted(fused.items(), key=lambda kv: -kv[1])][:k]

print(f"hybrid recall@5: {recall_at_k(rrf):.3f}")

hybrid recall@5: 0.944


## Strategy 4 — hybrid, then rerank

A cross-encoder scores query and chunk *together* rather than separately. Far
more accurate, and only affordable because it runs over a shortlist.

In [6]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(e, k=5, pool=20):
    cand = rrf(e, k=pool, pool=pool)
    texts = [chunks.loc[chunks.chunk_id == c, "text"].iloc[0] for c in cand]
    scores = reranker.predict([(e["question"], t) for t in texts])
    return [c for _, c in sorted(zip(scores, cand), key=lambda p: -p[0])][:k]

print(f"reranked recall@5: {recall_at_k(rerank):.3f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

reranked recall@5: 0.944


## The comparison

In [7]:
board = pd.DataFrame([
    {"strategy": "dense (vector)",       "recall@5": round(recall_at_k(dense), 3)},
    {"strategy": "BM25 (keyword)",       "recall@5": round(recall_at_k(keyword), 3)},
    {"strategy": "hybrid (RRF)",         "recall@5": round(recall_at_k(rrf), 3)},
    {"strategy": "hybrid + rerank",      "recall@5": round(recall_at_k(rerank), 3)},
]).sort_values("recall@5", ascending=False)
board

,strategy,recall@5
0,dense (vector),0.944
1,BM25 (keyword),0.944
2,hybrid (RRF),0.944
3,hybrid + rerank,0.944


## Where each strategy fails

An aggregate hides which questions are broken. Look per question.

In [8]:
rows = []
for e, qv in zip(EVAL, qvecs):
    if not e["gold_docs"]:
        continue
    r = {"qid": e["qid"], "hops": e["hops"], "question": e["question"][:52]}
    for name, fn in [("dense", lambda: docs_of(dense(qv))),
                     ("bm25", lambda: docs_of(keyword(e))),
                     ("hybrid", lambda: docs_of(rrf(e))),
                     ("rerank", lambda: docs_of(rerank(e)))]:
        r[name] = round(len(set(e["gold_docs"]) & fn()) / len(e["gold_docs"]), 2)
    rows.append(r)
per_q = pd.DataFrame(rows)
per_q

,qid,hops,question,dense,bm25,hybrid,rerank
0,q01,1,What is the maximum gearbox oil temperature during c,1.0,1.0,1.0,1.0
1,q02,1,How often must a turbine older than twelve years be,1.0,1.0,1.0,1.0
2,q03,1,What caused the shutdown of turbine WHL-031?,1.0,1.0,1.0,1.0
3,q04,1,Which site had the highest failure rate in Q2 2026?,1.0,1.0,1.0,1.0
4,q05,2,Who is the site manager responsible for the site wit,0.5,0.5,0.5,0.5
5,q06,2,Was the inspection deferral before incident 2026-021,1.0,1.0,1.0,1.0
6,q07,2,"What vibration level triggers immediate shutdown, an",1.0,1.0,1.0,1.0
7,q08,1,How many operational turbines are there at Whitelee?,1.0,1.0,1.0,1.0
8,q09,1,What approval is needed to return a turbine to servi,1.0,1.0,1.0,1.0


### Read the `hops` column

Look at where the scores drop. The multi-hop questions need facts from two
documents, and no single-shot strategy reliably retrieves both — because the
question embeds close to one of them and not the other.

**That is not a tuning problem.** It is the limit that agentic retrieval exists
to address, and notebook 03 shows the loop clearing it.

In [9]:
print(per_q.groupby("hops")[["dense","bm25","hybrid","rerank"]].mean().round(3))
print("\nsingle-hop questions are largely solved; multi-hop are not.")

      dense   bm25  hybrid  rerank
hops                              
1     1.000  1.000   1.000   1.000
2     0.833  0.833   0.833   0.833

single-hop questions are largely solved; multi-hop are not.


## Index recall — the number nobody measures

Everything above assumed the index returned what exact search would. Check it.

In [10]:
try:
    import psycopg2
    conn = psycopg2.connect("host=localhost port=5433 dbname=aiml user=aiml password=aiml")
    conn.autocommit = True
    rows = []
    for ef in [4, 10, 40, 200]:
        agree = []
        for qv in qvecs:
            exact = set(chunks.iloc[np.argsort(-(vecs @ qv))[:5]]["chunk_id"])
            with conn.cursor() as cur:
                cur.execute("SET hnsw.ef_search = %s", (ef,))
                cur.execute("SELECT chunk_id FROM chunks ORDER BY embedding <=> %s::vector LIMIT 5",
                            (list(map(float, qv)),))
                approx = {r[0] for r in cur.fetchall()}
            agree.append(len(exact & approx) / 5)
        rows.append({"ef_search": ef, "recall@5_vs_exact": round(float(np.mean(agree)), 3)})
    display(pd.DataFrame(rows))
    print("\nRaise ef_search before rebuilding the index. It costs latency, not a rebuild.")
except Exception as ex:
    print(f"pgvector not reachable ({ex.__class__.__name__}) — run notebook 01 first")

,ef_search,recall@5_vs_exact
0,4,1.0
1,10,1.0
2,40,1.0
3,200,1.0



Raise ef_search before rebuilding the index. It costs latency, not a rebuild.


---

## What this established

- Four strategies measured against gold labels, not impressions
- Where each one fails, per question, rather than as one average
- Multi-hop questions are not a tuning problem
- Index recall measured against exact search across `ef_search` settings

## Your turn

1. Re-run notebook 01 with `chunk_fixed`, then re-run this notebook. How much
   recall does bad chunking cost?
2. Set `pool` to 5 in `rrf`. Does reranking still help?
3. Add a question of your own with a gold label and see which strategies find it.